# Train with Colab

In [ ]:
from huggingface_hub import login
!pip install --upgrade transformers
import transformers
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True" #Set environment variables to reduce memory fragmentation


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 24.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.44.2
    Uninstalling transformers-4.44.2:
      Successfully uninstalled transformers-4.44.2


###MWP generation

### Load the tokennizer

In [ ]:
!pip install python-dotenv

In [ ]:
from dotenv import load_dotenv

env_path = '/content/token.env'

load_dotenv(env_path)

token = os.getenv('HUGGINGFACE_TOKEN')

# check of token loading successfully
if token:
    print("Token successfully retrieved")
else:
    print("Token not found")

Token successfully retrieved


### Load the model

In [ ]:
# Set environment variable to enable CUDA device-side assertion debugging
os.environ['TORCH_USE_CUDA_DSA'] = '1'

# Load the model
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Load tokenizer
model = AutoModelForCausalLM.from_pretrained(model_id, use_auth_token=token)
model.to("cuda")  # Load onto the GPU
tokenizer = AutoTokenizer.from_pretrained(model_id, use_auth_token=token)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Print the results if load successfully
print("Model and tokenizer loaded successfully!")

/usr/local/lib/python3.10/dist-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

/usr/local/lib/python3.10/dist-packages/transformers/models/auto/tokenization_auto.py:796: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Model and tokenizer loaded successfully!


In [ ]:
torch.cuda.empty_cache()
!nvidia-smi

Fri Oct 11 08:17:17 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:00:04.0 Off |                    0 |
| N/A   33C    P0              54W / 400W |  31065MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

### Load the dataset and set the new name for fine-tune model

In [ ]:
# The model that you want to train from the Hugging Face hub
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# The instruction dataset to use
dataset_name = "/content/sample_data/MWP/test.csv"

# Fine-tuned model name
new_model = "Fine_tune_meta-llama/Meta-Llama-3.1-8B-Instruct"

### Handling the testing dataset

In [ ]:
import pandas as pd
# Load the dataset with a different encoding
df_test = pd.read_csv('/content/test.csv', encoding='ISO-8859-1')

# Display the first few rows of the dataset
df_test.head()

,prompt
0,<s> [INST] Think you are a maths teacher. Crea...
1,<s> [INST] Think you are a maths teacher. Crea...
2,<s> [INST] Think you are a maths teacher. Crea...
3,<s> [INST] Think you are a maths teacher. Crea...
4,<s> [INST] Think you are a maths teacher. Crea...


In [ ]:
# Transfert dataset into hugginface Dataset
!pip install datasets
from datasets import Dataset

# Create a Hugging Face dataset
dataset = Dataset.from_pandas(df_test)

# Display a few samples
print(dataset)

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.5 MB/s eta 0:00:00
Dataset({
    features: ['prompt'],
    num_rows: 999
})


# Handling with Datasets

In [ ]:
import re

# Step 1: Load the dataset and split prompt/labels
df_test = pd.read_csv('/content/test.csv', encoding='ISO-8859-1')

prompts = []
labels = []

# Use regular expressions to match "Generated questions" or "Generated?questions"
for index, row in df_test.iterrows():
    # Match the prompt part
    prompt_match = re.search(r'<s>\s\[INST\](.*?)\[\/INST\]\s<\/s>', row['prompt'], re.DOTALL)

    # Match the part after "Generated questions" or "Generated?questions"
    labels_match = re.search(r'Generated[\?]?questions:\s*(.*)', row['prompt'], re.DOTALL)

    prompt = prompt_match.group(0).strip() if prompt_match else ''
    label = labels_match.group(1).strip() if labels_match else ''  # Ensure the generated question part is correctly extracted

    prompts.append(prompt)
    labels.append(label)

# Create a new DataFrame with separated prompt and labels
df_split = pd.DataFrame({
    'prompt': prompts,
    'labels': labels  # The original labels
})

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df_split)

# Load the tokenizer (adjust the model name to your specific model)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Step 3: Tokenization function
def tokenize_function(examples):
    # Tokenize the prompt to generate input_ids and attention_mask with max_length=80
    inputs = tokenizer(examples['prompt'], truncation=True, padding="max_length", max_length=80, return_tensors="pt")

    # Tokenize the labels to generate the target sequence with max_length=80
    outputs = tokenizer(examples['labels'], truncation=True, padding="max_length", max_length=80, return_tensors="pt")

    # Return input_ids, attention_mask, and labels
    return {
        'input_ids': inputs['input_ids'].squeeze(),  # Squeeze to remove extra dimension
        'attention_mask': inputs['attention_mask'].squeeze(),
        'labels': outputs['input_ids'].squeeze()  # Labels as target sequence
    }

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Set dataset format to PyTorch tensors
tokenized_dataset.set_format("torch")

# Print the first few tokenized samples
for i in range(2):
    print(f"Original Prompt: {df_split['prompt'][i]}")
    print(f"Tokenized Input IDs: {tokenized_dataset[i]['input_ids']}")
    print(f"Attention Mask: {tokenized_dataset[i]['attention_mask']}")
    print(f"Labels: {tokenized_dataset[i]['labels']}\n")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/999 [00:00<?, ? examples/s]

Original Prompt: <s> [INST] Think you are a maths teacher. Create math word problems satisfying the following requirements:
  [
      Grade: 1,
      Section: Addition & Subtraction(Within 20),
      Number of questions: 1

  ]
 [/INST] </s>
Tokenized Input IDs: tensor([  101,  1026,  1055,  1028,  1031, 16021,  2102,  1033,  2228,  2017,
         2024,  1037,  8785,  2015,  3836,  1012,  3443,  8785,  2773,  3471,
        17087,  1996,  2206,  5918,  1024,  1031,  3694,  1024,  1015,  1010,
         2930,  1024,  2804,  1004,  4942,  6494,  7542,  1006,  2306,  2322,
         1007,  1010,  2193,  1997,  3980,  1024,  1015,  1033,  1031,  1013,
        16021,  2102,  1033,  1026,  1013,  1055,  1028,   102,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0])
Attention Mask: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1,

In [ ]:
import transformers
print(transformers.__version__)

4.45.2


### Training preparation

In [ ]:
#!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7

In [ ]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 trl==0.4.7 --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.5/92.5 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 7.4 MB/s eta 0:00:00


In [ ]:
from transformers import BitsAndBytesConfig

# Set 4-bit quantization configuration
use_4bit = True  # Enable 4-bit quantization
use_nested_quant = True  # Enable nested quantization
bnb_4bit_quant_type = "nf4"  # Assuming  using the "nf4" quantization type
bnb_4bit_compute_dtype = 'float16'  # Assuming using float16

# Dynamically retrieve the compute data type
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

# Check if the GPU supports bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Apply the quantization settings to the loaded model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant
)

# Modify the already loaded model configuration
model.config.quantization_config = bnb_config
model.config.use_cache = False  # Set to False if you need to disable caching
model.config.pretraining_tp = 1  # Adjust tensor parallelism degree if needed

# Print confirmation message
print("4-bit quantization and additional settings have been applied to the loaded model!")

Your GPU supports bfloat16: accelerate training with bf16=True
4-bit quantization and additional settings have been applied to the loaded model!


In [ ]:
torch.cuda.empty_cache()

In [ ]:
from transformers import BitsAndBytesConfig
from transformers import AdamW

################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 32

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.1

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False  # Updated based on the second block

# Dynamically retrieve the compute data type
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

# Check if the GPU supports bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Apply the quantization settings to the loaded model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant
)

# Modify the already loaded model configuration
#model.config.quantization_config = bnb_config
model.config.quantization_config = None
model.config.use_cache = False  # Set to False if need to disable caching
model.config.pretraining_tp = 1  # Adjust tensor parallelism degree if needed

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs - updated to 3-5 for better learning
num_train_epochs = 5

# Enable fp16/bf16 training (set bf16 to True with an A100)
#fp16 = False
fp16 = True
bf16 = False

# Batch size per GPU for training
#per_device_train_batch_size = 2
per_device_train_batch_size = 1

# Batch size per GPU for evaluation
#per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 8

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
# optim = "paged_adamw_32bit"

# Use AdamW optimiser
optimizer = AdamW(
    model.parameters(),
    lr=2e-4,  # Learning rate
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.01
)

# Learning rate schedule (constant a bit better than cosine)
lr_scheduler_type = "constant"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 25

# Log every X updates steps
logging_steps = 25

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = 512

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

# Print confirmation message
print("4-bit quantization and additional settings have been applied to the loaded model!")

Your GPU supports bfloat16: accelerate training with bf16=True


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


4-bit quantization and additional settings have been applied to the loaded model!


### Train model

In [ ]:
!pip install transformers[torch] accelerate>=0.26.0
import accelerate
print(accelerate.__version__)

0.34.2


In [ ]:
#!pip install --upgrade transformers
# os.kill(os.getpid(), 9)
#!pip install --upgrade accelerate datasets
from transformers import TrainingArguments, Trainer

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    #per_device_train_batch_size=per_device_train_batch_size,
    #gradient_accumulation_steps=gradient_accumulation_steps,
    #optim=optim,
    gradient_accumulation_steps=8,
    per_device_train_batch_size=1,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    #fp16=fp16,
    fp16=True,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

In [ ]:
if model.config.quantization_config is None:
    model.config.quantization_config = bnb_config
    print("Manually applied quantization config")
else:
    print("Quantization config already set")

Manually applied quantization config


In [ ]:
if torch.cuda.is_available():
    print("CUDA is available. Using GPU.")
else:
    print("CUDA is not available. Using CPU.")

CUDA is available. Using GPU.


In [ ]:
# Check GPU
!nvidia-smi

device = torch.device("cuda")
model.to(device)
inputs = tokenizer("Example input text", return_tensors="pt").to(device)
inputs.pop("token_type_ids")
outputs = model.generate(**inputs)

Fri Oct 11 08:17:40 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off | 00000000:00:04.0 Off |                    0 |
| N/A   33C    P0              54W / 400W |  31065MiB / 40960MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1220: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


In [ ]:
# Check available GPU
print(torch.cuda.is_available())

print(torch.cuda.device_count())

print(torch.cuda.get_device_name(0))

True
1
NVIDIA A100-SXM4-40GB


In [ ]:
torch.cuda.empty_cache()

In [ ]:
!pip install --upgrade transformers trl
#from trl import SFTTrainer
from transformers import Trainer
os.environ['TORCH_USE_CUDA_DSA'] = '1'
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    args=training_arguments,
    optimizers=(optimizer, None)
)

# Enable gradient anomaly detection
torch.autograd.set_detect_anomaly(True)

trainer.train()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.4.7
    Uninstalling trl-0.4.7:
      Successfully uninstalled trl-0.4.7


/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.mlu.amp.GradScaler(**kwargs)


OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 39.56 GiB of which 110.81 MiB is free. Process 31797 has 39.45 GiB memory in use. Of the allocated memory 38.94 GiB is allocated by PyTorch, and 10.69 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

# 解决方法

降低 batch_size（首选）

释放显存（torch.cuda.empty_cache()）

启用 fp16 训练

启用 gradient_checkpointing

避免显存碎片化（expandable_segments:True）